## Env

In [ ]:
pwd

In [ ]:
!conda env

In [2]:
from __future__ import annotations

import os

# Must be set before importing torch. Adjust to a free GPU on this machine.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import json
import subprocess
import sys
import tqdm
import textwrap
from pathlib import Path
from pprint import pprint
from collections import defaultdict, OrderedDict, Counter

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 9

In [3]:
HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ROOT = HERE.parent
elif HERE.name == "manim_play":
    ROOT = HERE
elif (HERE / "benchmark" / "manim_play").is_dir():
    ROOT = HERE / "benchmark" / "manim_play"
else:
    ROOT = Path("/workspace-SR004.nfs2/rahmatullaev/rand_exps/PoC_for_rand_ideas/manim_play")

# External deps reused in place from abdullaeva's environment (not copied into this repo).
SPECTRA_ROOT = Path("/workspace-SR004.nfs2/abdullaeva/video-llm/Streaming/SPECTRA")
MINICPM_CHECKPOINT = Path("/workspace-SR004.nfs2/abdullaeva/video-llm/Streaming/MiniCPM-o-4_5")

ANN = ROOT / "data" / "annotations"
BUNDLES = ROOT / "data" / "bundles"
VIDEOS = ROOT / "data" / "videos"
RUNS = ROOT / "data" / "runs"
CLIPS = ROOT / "data" / "clips"
OUTPUT_PATH = ROOT / "outputs"

print(f"manim_play: {ROOT} --> exists? {ROOT.exists()}")
print(f"VIDEOS path: {VIDEOS} --> exists? {VIDEOS.exists()}")
print(f"SPECTRA: {SPECTRA_ROOT} --> exists? {SPECTRA_ROOT.exists()}")
print(f"MiniCPM checkpoint: {MINICPM_CHECKPOINT} --> exists? {MINICPM_CHECKPOINT.exists()}")


manim_play: /home/jovyan/shares/SR004.nfs2/rahmatullaev/rand_exps/PoC_for_rand_ideas/manim_play --> exists? True
VIDEOS path: /home/jovyan/shares/SR004.nfs2/rahmatullaev/rand_exps/PoC_for_rand_ideas/manim_play/data/videos --> exists? True
SPECTRA: /workspace-SR004.nfs2/abdullaeva/video-llm/Streaming/SPECTRA --> exists? True
MiniCPM checkpoint: /workspace-SR004.nfs2/abdullaeva/video-llm/Streaming/MiniCPM-o-4_5 --> exists? True


In [4]:
HERE

PosixPath('/home/jovyan/shares/SR004.nfs2/rahmatullaev/rand_exps/PoC_for_rand_ideas/manim_play/notebooks')

In [5]:
!ls {VIDEOS}

3d6DsjIBzJ4  Cld0p3a43fU  GlYgs6v2YfU  PFDu9oVAE-g  XFDM1ip5HdU  l6DKRf-fAAM


In [6]:
VIDEO_NAMES = [v.name for v in VIDEOS.glob("*") if v.is_dir()]
print(f"Total: {len(VIDEO_NAMES)} videos processed.")
for v in VIDEO_NAMES:
    print(f"\t-  {v}")

Total: 6 videos processed.
	-  l6DKRf-fAAM
	-  3d6DsjIBzJ4
	-  GlYgs6v2YfU
	-  PFDu9oVAE-g
	-  Cld0p3a43fU
	-  XFDM1ip5HdU


In [7]:
def load_samples(video_id):
    path = ANN / f"{video_id}.jsonl"
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


def does_bundle_exist(video_id):
    if (BUNDLES / video_id / "bundle.json").exists():
        return True
    return False

def does_annotation_exist(video_id):
    ann = ANN / f"{video_id}.jsonl"
    if ann.exists():
        return True
    return False

def load_bundle(video_id):
    return json.loads((BUNDLES / video_id / "bundle.json").read_text(encoding="utf-8"))


def load_info(video_id):
    p = VIDEOS / video_id / f"{video_id}.info.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.is_file() else {}

In [8]:
VALID_VIDEO_NAMES = [v for v in VIDEO_NAMES if does_bundle_exist(v) and does_annotation_exist(v)]
print(f"Total annotated: {len(VALID_VIDEO_NAMES)} videos.")
for v in VALID_VIDEO_NAMES:
    print(f"\t-  {v}")

Total annotated: 3 videos.
	-  l6DKRf-fAAM
	-  3d6DsjIBzJ4
	-  GlYgs6v2YfU


## Explore

In [9]:
VALID_VIDEO_NAMES

['l6DKRf-fAAM', '3d6DsjIBzJ4', 'GlYgs6v2YfU']

In [10]:
VIDEO_ID = VALID_VIDEO_NAMES[1]

samples = load_samples(VIDEO_ID)
bundle_content = load_bundle(VIDEO_ID)
info = load_info(VIDEO_ID)
video_filepath = VIDEOS / VIDEO_ID / f"{VIDEO_ID}.mp4"

print(f"video   : {bundle_content['title']}")
print(f"youtube : {bundle_content['video_id']}   uploaded {bundle_content.get('upload_date')}   duration {bundle_content['duration']:.1f}s")
print(f"source  : {bundle_content.get('source_code')}   chapters {len(bundle_content['chapters'])}   "
      f"transcript cues {len(bundle_content['transcript'])}   manim formulas {len(bundle_content['formulas'])}")
print(f"samples : {len(samples)}   triggers {sum(len(s['triggers']) for s in samples)}   "
      f"required {sum(1 for s in samples for t in s['triggers'] if t['required'])}  "
      f"not required {sum(1 for s in samples for t in s['triggers'] if not t['required'])}")
if info:
    print(f"yt-dlp  : format={info.get('format_id') or info.get('format')}  "
          f"res={info.get('width')}x{info.get('height')}  fps={info.get('fps')}")

video   : Taylor series | Chapter 11, Essence of calculus
youtube : 3d6DsjIBzJ4   uploaded 20170507   duration 1339.0s
source  : _2017/eoc/chapter10.py   chapters 5   transcript cues 1016   manim formulas 69
samples : 18   triggers 29   required 18  not required 11
yt-dlp  : format=18  res=640x360  fps=30


In [11]:
samples[0]

{'id': '3b1b__3d6DsjIBzJ4__q01',
 'video': {'youtube_id': '3d6DsjIBzJ4',
  'title': 'Taylor series | Chapter 11, Essence of calculus',
  'release_date': '2017-05-07',
  'duration': 1339.0,
  'source_code': '_2017/eoc/chapter10.py'},
 'question': 'Once the displayed pendulum diagram has all three labels R, θ, and its vertical height expression visible, suppose the angle shown were doubled while R stayed fixed. Give the exact new height expression and its quadratic small-angle approximation in θ.',
 'question_time': 35.0,
 'question_time_note': 'The pendulum has appeared, but its geometric labels and height expression have not yet been completed.',
 'act': 'counterfact',
 'temporal_rel': 'pre',
 'reasoning_level': 'L3',
 'template': 'counterfactual on-appearance',
 'triggers': [{'idx': 0,
   'required': True,
   't_start': 51.0,
   't_end': 79.8,
   't_ideal': 52.0,
   'response': 'The exact height would be R(1 − cos(2θ)). Using the small-angle form visible later in the same pendulum seq

In [12]:
for si, s in enumerate(samples):
    if s['validation']['status'] != "validated":
        print(f"#{si} not validates QA:")
        pprint(s['validation'])
        print(f"====="*10, "\n")

#3 not validates QA:
{'blind_solved': False,
 'frozenframe_solved': None,
 'notes': 'Original L3 was rejected because the author revealed c2=−1/2 within '
          't_end+10. After participant-visible rewrite to a doubled-curvature '
          'counterfactual, complete fresh round 3 yielded semiblind t_val=273, '
          'all modality probes solved, and full_sample.3/full_window0.3 PASS. '
          'Final conservative start is 272.0. | 2026-09-08 ladder (2.3, gap '
          "167.2 s): the when-clause 'As soon as the blue target's "
          'second-derivative row and the still-free quadratic term are '
          "simultaneously readable' contributes no rubric item - the pendulum "
          'scene prints cos(theta) ~ 1 - theta^2/2 at 77.0, from which the '
          'target curvature -1 follows in one step, and the doubling rule is '
          'supplied by the question itself - so it is EMPTY. Its referent (the '
          'second-derivative row) does not exist at t_val 77.0, so 

In [13]:
for si, s in enumerate(samples):
    print(f"#{si} Q at {s['question_time']}, lvl: {s['reasoning_level']}, act: {s['act']};\nQ: {s['question']}")
    triggers = s['triggers']
    print(f"Got {len(triggers)} triggers:")
    for ti, t in enumerate(triggers):
        print(f"\t- [{t['t_start']}-{t['t_end']}] with A: {t['response']}")
    print("\n=======\n")

#0 Q at 35.0, lvl: L3, act: counterfact;
Q: Once the displayed pendulum diagram has all three labels R, θ, and its vertical height expression visible, suppose the angle shown were doubled while R stayed fixed. Give the exact new height expression and its quadratic small-angle approximation in θ.
Got 2 triggers:
	- [51.0-79.8] with A: The exact height would be R(1 − cos(2θ)). Using the small-angle form visible later in the same pendulum sequence, cos(2θ) ≈ 1 − (2θ)^2/2, so the height is approximately 2Rθ^2.
	- [499.0-507.0] with A: The exact height would be R(1 − cos(2θ)). Using the displayed cosine expansion, its quadratic small-angle approximation is 2Rθ².


#1 Q at 130.3, lvl: L2, act: counterfact;
Q: Use the blue target function now on the screen and its displayed center. Identify both, then reflect that target across the x-axis while keeping the same center and degree; determine all three coefficients and give the resulting polynomial.
Got 2 triggers:
	- [130.3-139.5] with A: For −

In [14]:
Counter([s['reasoning_level'] for si, s in enumerate(samples)])

Counter({'L1': 7, 'L3': 6, 'L2': 5})

In [15]:
samples_by_lvl = defaultdict(list)
for si, s in enumerate(samples):
    samples_by_lvl[s['reasoning_level']].append(s)

for lvl_name, lvl_samples in samples_by_lvl.items():
    print(f"{lvl_name}: {len(lvl_samples)} samples")

L3: 6 samples
L2: 5 samples
L1: 7 samples


In [16]:
def overview(samples) -> pd.DataFrame:
    rows = []
    for i, s in enumerate(samples):
        req = [t for t in s["triggers"] if t["required"]]
        first_req = req[0]
        rows.append({
            "i": i,
            "id": s["id"].split("__")[-1],
            "template": s["template"],
            "act": s["act"],
            "rel": s["temporal_rel"],
            "lvl": s["reasoning_level"],
            "n_trig": len(s["triggers"]),
            "n_req": len(req),
            "n_dis": len(s.get("distractor_windows") or []),
            "q_time": round(s["question_time"], 1),
            "wait_s": round(first_req["t_start"] - s["question_time"], 1),
            "span_s": round(max(t["t_end"] for t in s["triggers"]) - s["question_time"], 1),
            "audio_claim": s.get("audio_dependency_claim"),
            "status": s["validation"]["status"],
        })
    return pd.DataFrame(rows).set_index("i")


df = overview(samples)
display(df)

print("\ncoverage  act × temporal_rel × reasoning_level")
display(pd.crosstab([df["act"], df["rel"]], df["lvl"], margins=True))
print("templates:")
print(df["template"].value_counts().to_string())

,id,template,act,rel,lvl,n_trig,n_req,n_dis,q_time,wait_s,span_s,audio_claim,status
i,,,,,,,,,,,,,
0,q01,counterfactual on-appearance,counterfact,pre,L3,2,1,0,35.0,16.0,472.0,unnecessary,validated
1,q02,counterfactual on-appearance,counterfact,at,L2,2,1,0,130.3,9.7,32.0,unnecessary,validated
2,q03,coefficient-to-geometry grounding,ground,at,L1,5,1,0,141.0,88.0,572.0,unnecessary,validated
3,q04,counterfactual on-appearance,counterfact,at,L2,2,1,0,244.2,27.8,64.3,unnecessary,draft
4,q05,screen-specific numerical verification,verify,pre,L3,1,1,0,324.1,8.9,17.6,unnecessary,validated
5,q06,prediction-ahead,predict,pre,L3,1,1,0,443.5,6.9,22.5,unnecessary,validated
6,q07,counterfactual on-appearance,counterfact,pre,L2,1,1,0,510.9,16.1,38.5,unnecessary,validated
7,q08,counterfactual on-appearance,counterfact,at,L2,1,1,0,550.1,23.9,40.4,unnecessary,draft
8,q09,symbol-binding / notation tracking,ground,pre,L3,1,1,0,592.5,9.5,31.3,unnecessary,validated



coverage  act × temporal_rel × reasoning_level


lvl               L1  L2  L3  All
act         rel                  
counterfact at     0   3   0    3
            pre    0   1   2    3
ground      at     1   0   0    1
            post   1   0   0    1
            pre    1   0   1    2
monitor     pre    3   0   0    3
narrate     pre    1   0   0    1
predict     pre    0   1   2    3
verify      pre    0   0   1    1
All                7   5   6   18

templates:
template
counterfactual on-appearance                6
symbol-binding / notation tracking          3
prediction-ahead                            2
premature-answer trap                       2
screen-specific numerical verification      1
coefficient-to-geometry grounding           1
prediction-ahead from a prior derivation    1
cross-representation narration              1
screen-label grounding                      1


In [17]:
from IPython.display import Video, display


def resolve_video_path(video=None) -> Path:
    """Return the mp4 for a YouTube id, a filesystem path, or the current VIDEO_ID."""
    if video is None:
        video = VIDEO_ID
    path = Path(video)
    if not path.suffix:
        path = VIDEOS / str(video) / f"{video}.mp4"
    if not path.is_file():
        raise FileNotFoundError(f"Video not found: {path}")
    return path


def play_video(video=None, start=None, end=None, width=720):
    """Load an mp4 and show an HTML5 player.

    `video` is a path or YouTube id (defaults to VIDEO_ID). Pass `start`/`end`
    in seconds to cut a preview clip — full lectures are too large to embed.
    Clips are cached under data/clips/_preview/.
    """
    src = resolve_video_path(video)
    play_path = src
    note = src.name

    if start is not None or end is not None:
        if start is None or end is None:
            raise ValueError("pass both start and end, or neither")
        start, end = float(start), float(end)
        if end <= start:
            raise ValueError(f"end ({end}) must be greater than start ({start})")
        clip_dir = CLIPS / "_preview"
        clip_dir.mkdir(parents=True, exist_ok=True)
        stem = f"{src.stem}_{start:.1f}_{end:.1f}".replace(".", "p")
        play_path = clip_dir / f"{stem}.mp4"
        if not play_path.is_file() or play_path.stat().st_size < 1_000:
            subprocess.run(
                [
                    "ffmpeg", "-y", "-v", "error",
                    "-ss", f"{start:.3f}", "-i", str(src),
                    "-t", f"{end - start:.3f}",
                    "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23",
                    "-pix_fmt", "yuv420p",
                    "-c:a", "aac", "-ac", "2",
                    "-movflags", "+faststart",
                    str(play_path),
                ],
                check=True,
            )
        note = f"{src.name}  [{start:.1f}s, {end:.1f}s)"

    size_mb = play_path.stat().st_size / (1024 * 1024)
    embed = size_mb <= 40
    print(f"{note}  {size_mb:.1f} MB")
    display(Video(str(play_path), embed=embed, width=width, html_attributes="controls"))


# Opening of the current lecture. For a trigger window:
# t = samples[0]["triggers"][0]
# play_video(start=t["t_start"], end=t["t_end"])
# play_video(start=0, end=12)

In [18]:
sample = samples[0]
video_id = sample["video"]["youtube_id"]
video_path = VIDEOS / video_id / f"{video_id}.mp4"

In [19]:
sample['triggers']

[{'idx': 0,
  'required': True,
  't_start': 51.0,
  't_end': 79.8,
  't_ideal': 52.0,
  'response': 'The exact height would be R(1 − cos(2θ)). Using the small-angle form visible later in the same pendulum sequence, cos(2θ) ≈ 1 − (2θ)^2/2, so the height is approximately 2Rθ^2.',
  'rubric': ['the exact expression R(1 − cos(2θ)) is given',
   'the quadratic approximation 2Rθ^2 is given',
   'the factor of four inside the quadratic angle term, followed by the one-half factor, is handled correctly',
   'do NOT count R(1 − cos θ) or Rθ^2/2, which leave the displayed angle unchanged'],
  'trigger_modality': ['visual'],
  'evidence_span': [51.0, 68.9]},
 {'idx': 1,
  'required': False,
  't_start': 499.0,
  't_end': 507.0,
  't_ideal': 500.0,
  'response': 'The exact height would be R(1 − cos(2θ)). Using the displayed cosine expansion, its quadratic small-angle approximation is 2Rθ².',
  'rubric': ['the exact expression R(1 − cos(2θ)) is given',
   'the quadratic approximation 2Rθ² is given'

In [20]:
print(f"question_time = {sample['question_time']}")
for t in sample['triggers']:
    print(f"#{t['idx']}: start-end: {t['t_start']} - {t['t_end']}, evidence: {t['evidence_span']}")

question_time = 35.0
#0: start-end: 51.0 - 79.8, evidence: [51.0, 68.9]
#1: start-end: 499.0 - 507.0, evidence: [499.0, 500.0]


In [21]:
t = samples[0]["triggers"][0]
play_video(video_path, 
           start=t["t_start"], end=t["t_end"]
          )

3d6DsjIBzJ4.mp4  [51.0s, 79.8s)  1.1 MB


In [22]:
video_path

PosixPath('/home/jovyan/shares/SR004.nfs2/rahmatullaev/rand_exps/PoC_for_rand_ideas/manim_play/data/videos/3d6DsjIBzJ4/3d6DsjIBzJ4.mp4')

## Eval

In [23]:
# MiniCPM-o 4.5 duplex evaluation.
# Call `evaluate_sample_duplex(...)` once per QA sample.  Internally, duplex
# performs streaming prefill + generation once per incoming frame-time chunk.

In [24]:
# The SPECTRA adapter owns `as_duplex`, session preparation, and chunk handling.
SPECTRA_SRC = SPECTRA_ROOT / "src"
SPECTRA_SCRIPTS = SPECTRA_ROOT / "scripts"
for path in (SPECTRA_SRC, SPECTRA_SCRIPTS):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from spectra.models.minicpmo45_duplex import MiniCPMO45DuplexModel
from spectra.runtime.loop import run_record
from spectra.schema import BenchmarkRecord
from _eval_common import make_chunk_stream
from spectra.schema import  (
    JoinConfig, join_chunked_utterances
)
from spectra.metrics import (
    precision, purity, iou, sample_direction, alert_direction_rates,
    pauc, gpa, moc, uda, nearest_response_value, timeliness_score,
    decay_multiplier, speak_ratio
)

In [25]:
DUPLEX_SYSTEM_PROMPT = (
    "You are an AI assistant for real-time video stream question answering. "
    "Answer each question as soon as the video and audio observed so far "
    "contain enough information to support an answer. "
    "If a question cannot yet be answered, or no question is "
    "currently active keep silent."
)

model_config = {
    "model_name": "minicpmo45-duplex",
    "model_type": "minicpmo45_duplex",
    "checkpoint_path": str(MINICPM_CHECKPOINT),
    "device": "cuda",
    "session_mode": "per_question",
    # Use the Omni chunk stream so the question is injected as `OmniChunk.question_text`.
    # `load_audio=False` keeps this evaluation frame-only.
    "modality": "video_audio",
    "default_system_prompt": DUPLEX_SYSTEM_PROMPT,
    "inference": {
        "fps": 1.0,
        "load_audio": False,
        "max_new_speak_tokens": 256,
        "basic_window_high_tokens": 8_000,
        "basic_window_low_tokens": 6_000,
    },
}

model = MiniCPMO45DuplexModel.from_config(model_config)
print(f"Loaded {type(model).__name__} ({model.modality})")

`torch_dtype` is deprecated! Use `dtype` instead!
Encountered exception while importing librosa: No module named 'librosa'
Encountered exception while importing soundfile: No module named 'soundfile'


ImportError: This modeling file requires the following packages that were not found in your environment: librosa, soundfile. Run `pip install librosa soundfile`

In [ ]:
PRE_PAD_SECONDS = 10.0
POST_PAD_SECONDS = 20.0


def eval_stream_window(sample: dict) -> tuple[Path, float, float]:
    """Original mp4 plus the [clip_start, clip_end] window in video time."""
    video_id = sample["video"]["youtube_id"]
    video_path = VIDEOS / video_id / f"{video_id}.mp4"
    if not video_path.is_file():
        raise FileNotFoundError(f"Video not found: {video_path}")

    question_time = float(sample["question_time"])
    starts = [question_time]
    if sample["temporal_rel"] == "post":
        starts.extend(
            float((trigger.get("evidence_span") or [trigger["t_start"]])[0])
            for trigger in sample["triggers"]
        )
    clip_start = max(0.0, min(starts) - PRE_PAD_SECONDS)
    clip_end = min(
        float(sample["video"]["duration"]),
        max(float(trigger["t_end"]) for trigger in sample["triggers"]) + POST_PAD_SECONDS,
    )
    return video_path, clip_start, clip_end


def sample_to_duplex_record(sample: dict) -> BenchmarkRecord:
    """One duplex session per QA pair on the original lecture mp4.

    The question is injected at `question_time`. Frames are streamed from
    `clip_start` through `clip_end` (no ffmpeg crop). Response timestamps
    stay in original video time.
    """
    video_path, clip_start, clip_end = eval_stream_window(sample)
    return BenchmarkRecord(
        record_id=sample["id"],
        video_path=str(video_path),
        instruction=sample["question"],
        triggers=[],  # Native duplex decides when to speak; it is not force-triggered.
        source_benchmark="manim_play",
        task_name=sample["act"],
        question_inject_t=float(sample["question_time"]),
        stream_start_t=clip_start,
        stream_end_t=clip_end,
        inference_strategy="per_question",
        stop_on_first_response=False,
    )

In [ ]:
sample = samples[1]
sample

In [ ]:
sample.keys()

In [ ]:
sample['triggers']

In [ ]:
print(f"question_time = {sample['question_time']}")
print(f"triggers evidence_span = {[t.get('evidence_span') for t in sample['triggers']]}")
print(f"triggers t_start = {[t.get('t_start') for t in sample['triggers']]}")
print(f"video duration = {sample['video']['duration']}")

In [ ]:
video_path, clip_start, clip_end = eval_stream_window(sample)
record = sample_to_duplex_record(sample)
print(f"video_path         = {video_path}")
print(f"question_time      = {sample['question_time']}")
print(f"clip_start         = {clip_start}")
print(f"clip_end           = {clip_end}")
print(f"question_inject_t  = {record.question_inject_t}")
print(f"stream_start_t     = {record.stream_start_t}")
print(f"stream_end_t       = {record.stream_end_t}")

In [ ]:
responses = run_record(record, model, make_chunk_stream(record, model_config))
responses

In [ ]:
response_to_dict(responses[0])

In [ ]:
# play_video(video_path, start=clip_start, end=clip_end)

In [ ]:
print(f"Q: {sample['question']}")
print(f"GT A: {sample['triggers'][0]['response']}")

In [ ]:
def evaluate_sample_duplex(model, processor, sample: dict, model_config) -> list[dict]:
    """Run one complete duplex session for one QA pair and return predictions.

    `processor` is a chunk-stream factory such as `make_chunk_stream`. It
    supplies frames at `model_config['inference']['fps']`; the adapter performs
    one streaming prefill/generate cycle per chunk and returns all speech units.
    """
    record = sample_to_duplex_record(sample)
    responses = run_record(record, model, processor(record, model_config))
    return [
        response_to_dict(response)
        for response in responses
        if response.content.strip()
    ]


# One function call creates and runs one independent session for one QA pair.
# predictions = evaluate_sample_duplex(model, make_chunk_stream, samples[0], model_config)
# pprint(predictions)

In [ ]:
VIDEO_ID

In [ ]:
results_path = OUTPUT_PATH / f"minicpm_test_eval_video={VIDEO_ID}_updated_sys_prompt.json"

In [ ]:
n_new = 0
with results_path.open("w") as results_fh:
    pbar = tqdm.tqdm(samples, total=len(samples))
    for sample in pbar:
        # responses list 
        responses = evaluate_sample_duplex(model, make_chunk_stream, sample, model_config)
        sample['responses'] = responses
        results_fh.write(json.dumps(sample) + "\n")
        results_fh.flush()
        n_new += 1
        pbar.set_postfix(new=n_new, done=n_new)

In [ ]:
samples[0]

In [ ]:
# results_path = OUTPUT_PATH / "minicpm_test_eval_3_samples.json"

In [ ]:
# VIDEO_ID = VALID_VIDEO_NAMES[0]

# samples = load_samples(VIDEO_ID)

In [ ]:
# n_new = 0
# with results_path.open("w") as results_fh:
#     samples = 
#     pbar = tqdm.tqdm(samples, total=len(samples))
#     for sample in pbar:
#         # responses list 
#         responses = evaluate_sample_duplex(model, make_chunk_stream, sample)
#         sample['responses'] = responses
#         results_fh.write(json.dumps(sample) + "\n")
#         results_fh.flush()
#         n_new += 1
#         pbar.set_postfix(new=n_new, done=n_new)

## Score

In [ ]:
# The SPECTRA adapter owns `as_duplex`, session preparation, and chunk handling.
SPECTRA_SRC = SPECTRA_ROOT / "src"
SPECTRA_SCRIPTS = SPECTRA_ROOT / "scripts"
for path in (SPECTRA_SRC, SPECTRA_SCRIPTS):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from spectra.schema import (
    AlertGT, NarrationGT, StateTrackGT, DescribedEventGT, CustomGT, InstantQAGT, MultiturnDialogueGT,
    BenchmarkRecord, TriggerSpec, ModelResponse, OmniChunk, 
    TaskFamily, EventMoment, GroundTruth,
    response_from_dict, trigger_from_dict, response_to_dict
)
from spectra.schema import  (
    JoinConfig, join_chunked_utterances
)
from spectra.metrics import (
    precision, purity, iou, sample_direction, alert_direction_rates,
    pauc, gpa, moc, uda, nearest_response_value, timeliness_score,
    decay_multiplier, speak_ratio
)

In [ ]:
import json
import ast

def load_predictions(pred_path: str, skip_errors: bool = True):
    """
    Load predictions from file.

    Supports:
    - JSON (full file as a list/dict)
    - JSONL (one JSON object per line)
    - Fallback: line-based Python dict literals

    Returns:
        predictions: list
        failed_lines: list of (line_idx, line, error)
    """
    failed_lines = []

    # --- Try standard JSON first (single JSON doc) ---
    try:
        with open(pred_path, "r") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data, failed_lines
        else:
            return [data], failed_lines
    except json.JSONDecodeError:
        pass  # fallback to line-by-line parsing

    # --- Fallback: JSONL / python-evaluable lines ---
    predictions = []

    with open(pred_path, "r") as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            try:
                predictions.append(json.loads(line))
            except json.JSONDecodeError:
                try:
                    predictions.append(ast.literal_eval(line))
                except (ValueError, SyntaxError) as e:
                    if skip_errors:
                        failed_lines.append((idx, line, str(e)))
                        continue
                    else:
                        raise

    return predictions, failed_lines

In [ ]:
OUTPUT_PATH

In [ ]:
exp_filenames = [d for d in OUTPUT_PATH.glob("*.json") if (
    ('.ipynb_checkpoints' not in d.name) and d.is_file())]
exp_filenames.sort(key=lambda x: x.stat().st_mtime)
for i, f in enumerate(exp_filenames):
    print(f"{i}: {f.name}")

In [ ]:
res_file = exp_filenames[-1]
print(res_file.name)
predictions, failed_lines = load_predictions(res_file)

print(f"Failed: {len(failed_lines)}, loaded: {len(predictions)}")

In [ ]:
print(set([p['act'] for p in predictions]))
preds_by_act = defaultdict(list)
for p in predictions:
    preds_by_act[p['act']].append(p)

for act_name, act_p in preds_by_act.items():
    print(f"{act_name}: {len(act_p)}")

In [ ]:
for p in preds_by_act['alert']:
    print(len(p['responses']))

In [ ]:
preds_by_act['alert'][0]

In [ ]:
preds_by_act['alert'][0]['responses']

In [ ]:
join_config = JoinConfig(strategy='native_signal', max_utterance_span_s=10)
join_config

In [ ]:
[response_from_dict(p) for p in preds_by_act['alert'][0]['responses']]

In [ ]:
join_chunked_utterances([response_from_dict(p) for p in preds_by_act['alert'][0]['responses']], join_config)

## Timeline: predictions vs evidence

Compare loaded `predictions` against ground-truth evidence timestamps.

- **Overview** — one row per sample. Gold hatch = `evidence_span`, navy triangles = joined model utterances (scoring time = first chunk).
- **Per sample** — two lanes (ground truth vs model), with trigger windows, distractors, and utterance text.
- **GT vs pred** — each trigger's gold `response` (and rubric) is printed next to the model's joined utterances.

Green = required trigger window, pale green = optional, red = distractor, blue `>` = `question_time`, black `|` = `t_ideal`. Utterance bars span first→last chunk; the inverted triangle is the time scorers use.


In [ ]:
from matplotlib.lines import Line2D

COL_REQ, COL_OPT, COL_DIS, COL_Q = "#2e9e5b", "#a8dcb9", "#d1495b", "#3b6ea5"
COL_PRED, COL_EVID = "#1d4ed8", "#c9a227"


def _join_cfg(cfg=None):
    if cfg is not None:
        return cfg
    return JoinConfig(strategy="native_signal", max_utterance_span_s=10)


def joined_utterances(sample, cfg=None):
    raw = [response_from_dict(r) for r in (sample.get("responses") or [])]
    if not raw:
        return []
    joined = join_chunked_utterances(raw, _join_cfg(cfg))
    chunks_by_uid = defaultdict(list)
    for r in raw:
        key = r.utterance_id if r.utterance_id is not None else f"__t{r.t:.6f}"
        chunks_by_uid[key].append(r)
    out = []
    for u in joined:
        key = u.utterance_id if u.utterance_id is not None else f"__t{u.t:.6f}"
        chunks = chunks_by_uid.get(key) or [u]
        ts = [c.t for c in chunks]
        out.append({
            "t": float(u.t),
            "t_start": float(min(ts)),
            "t_end": float(max(ts)),
            "content": u.content or "",
            "n_chunks": len(chunks),
            "utterance_id": u.utterance_id,
            "chunk_times": sorted(float(t) for t in ts),
        })
    return out


def evidence_spans(sample):
    spans = []
    for trig in sample.get("triggers") or []:
        es = trig.get("evidence_span")
        if not es:
            continue
        start = float(es[0])
        end = float(es[1] if len(es) > 1 else es[0])
        spans.append((start, end, trig.get("idx")))
    return spans


def _first_evidence_t(sample):
    spans = evidence_spans(sample)
    if spans:
        return min(start for start, _, _ in spans)
    ideals = [
        float(t["t_ideal"]) for t in (sample.get("triggers") or [])
        if t.get("t_ideal") is not None
    ]
    return min(ideals) if ideals else None


def _duration(sample):
    video = sample.get("video") or {}
    if isinstance(video, dict) and video.get("duration"):
        return float(video["duration"])
    return None


def _collect_times(sample, utterances):
    times = [float(sample["question_time"])]
    for trig in sample.get("triggers") or []:
        times.extend([float(trig["t_start"]), float(trig["t_end"]), float(trig["t_ideal"])])
        times.extend(float(x) for x in (trig.get("evidence_span") or []))
    for d in sample.get("distractor_windows") or []:
        times.extend([float(d["t_start"]), float(d["t_end"])])
    for u in utterances:
        times.extend([u["t_start"], u["t_end"], u["t"]])
    return times, _duration(sample)


def _xlim(sample, utterances, zoom="auto"):
    times, dur = _collect_times(sample, utterances)
    if not times:
        return (0.0, dur or 1.0)
    lo, hi = min(times), max(times)
    span = max(hi - lo, 1.0)
    use_full = zoom == "full" or (zoom == "auto" and dur is not None and span > 0.4 * dur)
    if use_full and dur is not None:
        return (0.0, dur)
    pad = max(8.0, 0.08 * span)
    right = hi + pad
    if dur is not None:
        right = min(dur, right)
    return (max(0.0, lo - pad), right)


def _min_visible(width, xlim, frac=0.004):
    return max(width, frac * max(xlim[1] - xlim[0], 1.0))


def _draw_gt_lane(ax, sample, y, xlim, height=0.5, labels=False):
    for d in sample.get("distractor_windows") or []:
        ax.barh(
            y, d["t_end"] - d["t_start"], left=d["t_start"], height=height,
            color=COL_DIS, alpha=0.85, zorder=2,
        )
    for trig in sample.get("triggers") or []:
        w = max(float(trig["t_end"]) - float(trig["t_start"]), 0.0)
        ax.barh(
            y, _min_visible(w, xlim), left=trig["t_start"], height=height,
            color=COL_REQ if trig.get("required") else COL_OPT, zorder=3,
        )
        ax.plot(trig["t_ideal"], y, "|", color="black", ms=11, mew=1.3, zorder=5)
        if labels and trig.get("idx") is not None:
            ax.annotate(
                str(trig["idx"]), (trig["t_ideal"], y + height * 0.72),
                ha="center", va="bottom", fontsize=7, color="0.25",
            )
    for start, end, _idx in evidence_spans(sample):
        ax.barh(
            y, _min_visible(max(end - start, 0.0), xlim, frac=0.006),
            left=start, height=height * 0.55,
            color=COL_EVID, alpha=0.95, zorder=4, hatch="///",
        )
    ax.plot(sample["question_time"], y, ">", color=COL_Q, ms=8, zorder=6)


def _draw_pred_lane(ax, utterances, y, xlim, height=0.42):
    for i, u in enumerate(utterances):
        w = max(u["t_end"] - u["t_start"], 0.0)
        ax.barh(
            y, _min_visible(w, xlim), left=u["t_start"], height=height,
            color=COL_PRED, alpha=0.85, zorder=3,
        )
        ax.plot(
            u["t"], y, "v", color=COL_PRED, ms=7, zorder=5,
            markeredgecolor="white", markeredgewidth=0.4,
        )
        if len(utterances) <= 12:
            ax.annotate(str(i + 1), (u["t"], y + 0.32), ha="center", fontsize=7, color=COL_PRED)
        if 1 < u["n_chunks"] <= 40:
            ax.plot(
                u["chunk_times"], [y] * len(u["chunk_times"]),
                "|", color="white", ms=5, mew=0.8, zorder=4, alpha=0.75,
            )


def _timing_subtitle(sample, utterances):
    first_pred = utterances[0]["t"] if utterances else None
    first_ev = _first_evidence_t(sample)
    if first_pred is None:
        return "no model responses"
    if first_ev is None:
        return f"first pred @ {first_pred:.1f}s"
    delta = first_pred - first_ev
    if delta < -0.5:
        tag = f"early by {abs(delta):.1f}s"
    elif delta > 0.5:
        tag = f"late by {delta:.1f}s"
    else:
        tag = "on time"
    return f"first pred @ {first_pred:.1f}s   evidence @ {first_ev:.1f}s   {tag}"


def plot_prediction_timeline(sample, cfg=None, zoom="auto", figsize=(13, 3.1)):
    cfg = _join_cfg(cfg)
    utterances = joined_utterances(sample, cfg)
    xlim = _xlim(sample, utterances, zoom=zoom)
    fig, ax = plt.subplots(figsize=figsize)
    ax.axhspan(0.52, 1.48, color="0.96", zorder=0)
    ax.axhspan(-0.48, 0.48, color="0.96", zorder=0)
    _draw_gt_lane(ax, sample, y=1.0, xlim=xlim, labels=True)
    _draw_pred_lane(ax, utterances, y=0.0, xlim=xlim)
    ax.set_xlim(*xlim)
    ax.set_ylim(-0.7, 1.75)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["model", "ground truth"])
    ax.set_xlabel("time in video, s")
    ax.set_title(
        f"{sample.get('id', '?')}  ·  {sample.get('act', '?')} / {sample.get('template', '?')}\n"
        + _timing_subtitle(sample, utterances),
        fontsize=10, loc="left",
    )
    ax.legend(
        handles=[
            Patch(color=COL_REQ, label="required window"),
            Patch(color=COL_OPT, label="optional window"),
            Patch(facecolor=COL_EVID, hatch="///", edgecolor="0.3", label="evidence_span"),
            Patch(color=COL_DIS, label="distractor"),
            Line2D([0], [0], marker=">", color=COL_Q, linestyle="None", ms=8, label="question_time"),
            Line2D([0], [0], marker="|", color="black", linestyle="None", ms=10, label="t_ideal"),
            Patch(color=COL_PRED, label="model utterance"),
            Line2D([0], [0], marker="v", color=COL_PRED, linestyle="None", ms=7, label="scoring t"),
        ],
        loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=4, frameon=False, fontsize=8,
    )
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    return utterances


def prediction_overview(samples, cfg=None, figsize=None):
    cfg = _join_cfg(cfg)
    n = len(samples)
    if n == 0:
        print("no samples")
        return
    dur = max((_duration(s) or 0.0) for s in samples) or 1.0
    fig, ax = plt.subplots(figsize=figsize or (13, max(2.6, 0.42 * n + 1.9)))
    for i, sample in enumerate(samples):
        y = n - 1 - i
        ax.hlines(y, 0, dur, color="0.93", lw=7, zorder=1)
        _draw_gt_lane(ax, sample, y=y, xlim=(0.0, dur), height=0.55, labels=False)
        for u in joined_utterances(sample, cfg):
            ax.plot(
                u["t"], y, "v", color=COL_PRED, ms=6, zorder=6,
                markeredgecolor="white", markeredgewidth=0.3,
            )
    ax.set_xlim(0, dur)
    ax.set_yticks(range(n))
    ax.set_yticklabels([str(s.get("id", "?")).split("__")[-1] for s in reversed(samples)])
    ax.set_xlabel("time in video, s")
    ax.set_title("Overview: evidence vs model utterance times", fontsize=11, loc="left")
    ax.legend(
        handles=[
            Patch(color=COL_REQ, label="required window"),
            Patch(color=COL_OPT, label="optional"),
            Patch(facecolor=COL_EVID, hatch="///", edgecolor="0.3", label="evidence_span"),
            Patch(color=COL_DIS, label="distractor"),
            Line2D([0], [0], marker=">", color=COL_Q, linestyle="None", ms=7, label="question"),
            Line2D([0], [0], marker="v", color=COL_PRED, linestyle="None", ms=7, label="model utterance"),
        ],
        loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=6, frameon=False,
    )
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def timing_table(samples, cfg=None):
    rows = []
    for s in samples:
        uts = joined_utterances(s, cfg)
        ev = _first_evidence_t(s)
        fp = uts[0]["t"] if uts else None
        delta = None if fp is None or ev is None else round(fp - ev, 1)
        rows.append({
            "id": str(s.get("id", "")).split("__")[-1],
            "act": s.get("act"),
            "q_t": s.get("question_time"),
            "evidence_t": ev,
            "n_utt": len(uts),
            "first_pred": fp,
            "delta_s": delta,
        })
    return pd.DataFrame(rows)


def _wrap(text, prefix, width=108, hang=None):
    body = (text or "").replace("\n", " ").strip() or "∅"
    print(textwrap.fill(
        body, width=width, initial_indent=prefix,
        subsequent_indent=(" " * len(prefix)) if hang is None else hang,
    ))


def _utt_span(u):
    if u["n_chunks"] > 1:
        return f"{u['t_start']:.1f}–{u['t_end']:.1f}s"
    return f"{u['t']:.1f}s"


def _utt_in_window(utterances, t_start, t_end):
    inside = [u for u in utterances if t_start <= u["t"] < t_end]
    if inside:
        return inside[0], True
    if not utterances:
        return None, False
    return min(utterances, key=lambda u: abs(u["t"] - t_start)), False


def show_prediction_timeline(sample, cfg=None, zoom="auto"):
    utterances = plot_prediction_timeline(sample, cfg=cfg, zoom=zoom)
    print(
        f"{sample.get('id')}  ·  {sample.get('act')} / {sample.get('template')}"
        f"  ·  {sample.get('temporal_rel')}"
    )
    _wrap(sample.get("question") or "", "Q:  ", hang="    ")
    for trig in sample.get("triggers") or []:
        req = "required" if trig.get("required") else "optional"
        print(
            f"  trigger {trig.get('idx')}  {req}  "
            f"window [{trig['t_start']:.1f}, {trig['t_end']:.1f})  "
            f"t_ideal={trig['t_ideal']:.1f}  evidence={trig.get('evidence_span')}"
        )
        _wrap(trig.get("response") or "", "    GT:  ", hang="         ")
        for item in trig.get("rubric") or []:
            _wrap(item, "      - ", hang="        ")
    if not utterances:
        print("  (no model utterances)")
    else:
        print("  predicted utterances:")
        for i, u in enumerate(utterances, 1):
            _wrap(u["content"] or "∅", f"    {i}. [{_utt_span(u)}]  ", hang="         ")
    print("  GT vs pred:")
    for trig in sample.get("triggers") or []:
        t0, t1 = float(trig["t_start"]), float(trig["t_end"])
        u, inside = _utt_in_window(utterances, t0, t1)
        if u is None:
            tag = "no model utterance"
        elif inside:
            tag = f"in window @ {_utt_span(u)}"
        else:
            tag = f"no in-window utterance; nearest @ {_utt_span(u)}"
        print(f"    trigger {trig.get('idx')}  {tag}")
        _wrap(trig.get("response") or "", "      GT:    ", hang="             ")
        _wrap((u["content"] if u else None) or "(silence)", "      Pred:  ", hang="             ")
    return utterances


In [ ]:
timing_table(predictions, cfg=join_config)

In [ ]:
prediction_overview(predictions, cfg=join_config)

In [ ]:
for sample in predictions:
    show_prediction_timeline(sample, cfg=join_config)
    print(f"===="*20)

## Response quality and timing

Score loaded `predictions` with SPECTRA metrics. Each manim_play `act` maps to a SPECTRA family; families are never averaged together.

| `act` | SPECTRA family | What is scored |
|---|---|---|
| `alert` | `alert` | first-shot **precision** / **purity** / **IoU** / direction / `time_accuracy` (content judged when a judge is configured) |
| `monitor`, `narrate` | `narration` | **PAUC** over each required window (`ω=0.5`) |
| `ground`, `counterfact`, `predict`, `verify` | `described_event` | **timeliness** + **content_raw**, gated by the trigger window (`hard_window`) |

Extra diagnostics (not a SPECTRA family, reported beside the scores): **FAR** (utterances outside every trigger window), **trap rate** (share of `distractor_windows` that received an utterance), **silence rate** (share of required windows with no utterance).

Content scores need the SPECTRA LLM judge (`SPECTRA/.env`: `JUDGE_API_KEY`, `JUDGE_API_BASE`, `JUDGE_MODEL`). Timing metrics run even if the judge is unavailable.


In [ ]:
from spectra.aggregation.aggregate import aggregate_family_scores
from spectra.judges.content_judge import APIJudge, JudgeNotConfiguredError, configure_judge
from spectra.scorers.dispatch import score_record

ACT_FAMILY = {
    "alert": "alert",
    "monitor": "narration",
    "narrate": "narration",
    "ground": "described_event",
    "predict": "described_event",
    "verify": "described_event",
    "counterfact": "described_event",
}


def _load_spectra_env():
    candidates = [
        ROOT / ".env",
        ROOT / "notebooks" / ".env",
    ]
    env_path = next((p for p in candidates if p.is_file()), None)
    if env_path is None:
        return None
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val
    return env_path


def setup_spectra_judge():
    _load_spectra_env()
    model = os.getenv("JUDGE_MODEL", "")
    api_key = os.getenv("JUDGE_API_KEY") or os.getenv("OPENAI_API_KEY") or ""
    api_base = os.getenv("JUDGE_API_BASE") or os.getenv("OPENAI_BASE_URL") or ""
    if not model or not (api_key or api_base):
        print("[judge] not configured — content scores skipped; timing still runs")
        return False
    configure_judge(APIJudge(model=model, api_key=api_key or None, api_base=api_base or None))
    print(f"[judge] APIJudge  model={model}  base={api_base or 'https://api.openai.com/v1'}")
    return True


def _required_triggers(sample):
    return [t for t in (sample.get("triggers") or []) if t.get("required", True)]


def sample_to_score_record(sample, include_content=True):
    """Build a SPECTRA record whose triggers match this sample's act.

    When `include_content` is False (no judge), every act is scored as
    timing-only `alert` so precision/purity/IoU still run.
    """
    family = ACT_FAMILY.get(sample.get("act"), "described_event")
    if not include_content:
        family = "alert"
    triggers = []
    for trig in sample.get("triggers") or []:
        if family == "narration" and not trig.get("required", True):
            continue
        t0, t1 = float(trig["t_start"]), float(trig["t_end"])
        gold = trig.get("response") or ""
        if family == "alert":
            gt = AlertGT(
                event_time=t0,
                event_time_end=t1,
                all_event_times=[t0],
                all_event_times_end=[t1],
                expected_content=gold if include_content else None,
                tolerance=0.0,
                anchor_mode="strict",
            )
            used_family = "alert"
        elif family == "narration":
            gt = NarrationGT(t_start=t0, t_end=t1, gold=gold, omega=0.5, judge_max_score=2)
            used_family = "narration"
        else:
            gt = DescribedEventGT(
                event_time=t0,
                expected_content=gold,
                max_delay_s=max(t1 - t0, 1.0),
                judge_max_score=2,
                decay_mode="hard_window",
                window_start=t0,
                window_end=t1,
            )
            used_family = "described_event"
        triggers.append(TriggerSpec(
            t=t0,
            forced=False,
            task_family=used_family,
            ground_truth=gt,
            query=sample.get("question"),
            query_inject_t=float(sample["question_time"]),
        ))
    return BenchmarkRecord(
        record_id=sample["id"],
        video_path="",
        instruction=sample.get("question") or "",
        triggers=triggers,
        source_benchmark="manim_play",
        task_name=sample.get("act") or "",
    )


def extra_timing_diagnostics(sample, cfg=None):
    """FAR / trap / silence — manim_play extras next to SPECTRA family scores."""
    uts = joined_utterances(sample, cfg)
    windows = [
        (float(t["t_start"]), float(t["t_end"]))
        for t in (sample.get("triggers") or [])
    ]
    required = [
        (float(t["t_start"]), float(t["t_end"]))
        for t in _required_triggers(sample)
    ]
    distractors = [
        (float(d["t_start"]), float(d["t_end"]))
        for d in (sample.get("distractor_windows") or [])
    ]
    n_utt = len(uts)
    in_window = sum(1 for u in uts if any(a <= u["t"] < b for a, b in windows)) if windows else 0
    far = None if n_utt == 0 else 1.0 - (in_window / n_utt)
    trap_hits = sum(
        1 for a, b in distractors if any(a <= u["t"] < b for u in uts)
    )
    trap_rate = None if not distractors else trap_hits / len(distractors)
    silence = None if not required else sum(
        1 for a, b in required if not any(a <= u["t"] < b for u in uts)
    ) / len(required)
    latencies = []
    for a, b in required:
        inside = [u["t"] for u in uts if a <= u["t"] < b]
        if inside:
            latencies.append(min(inside) - a)
    t0, t1 = required[0] if required else (None, None)
    first_t = uts[0]["t"] if uts else None
    return {
        "n_utt": n_utt,
        "n_trig": len(required),
        "far": far,
        "trap_rate": trap_rate,
        "silence_rate": silence,
        "first_in_window_latency_s": None if not latencies else min(latencies),
        "mean_in_window_latency_s": None if not latencies else sum(latencies) / len(latencies),
        "timeliness": None if first_t is None or t0 is None else timeliness_score(
            first_t, t0, max(t1 - t0, 1.0)
        ),
        "sample_direction": sample_direction(first_t, t0, t1, 0.0) if t0 is not None else None,
    }


def _flatten_family(scores):
    rows = []
    for family, metrics in scores.items():
        entries = metrics if isinstance(metrics, list) else [metrics]
        for i, entry in enumerate(entries):
            if not isinstance(entry, dict):
                continue
            row = {"family": family, "trig": i}
            for k, v in entry.items():
                if isinstance(v, (dict, list)):
                    continue
                row[k] = v
            rows.append(row)
    return rows


def score_predictions(samples, cfg=None, include_content=True):
    cfg = _join_cfg(cfg)
    detail_rows, summary_rows, per_record = [], [], []
    errors = []
    for sample in samples:
        extra = extra_timing_diagnostics(sample, cfg)
        responses = [response_from_dict(r) for r in (sample.get("responses") or [])]
        family = ACT_FAMILY.get(sample.get("act"), "described_event")
        try:
            record = sample_to_score_record(sample, include_content=include_content)
            if not record.triggers:
                raise ValueError("no scoring triggers")
            scores = score_record(record, responses, cfg)
        except JudgeNotConfiguredError:
            record = sample_to_score_record(sample, include_content=False)
            scores = score_record(record, responses, cfg)
            errors.append((sample.get("id"), "judge not configured; timing-only"))
        except Exception as exc:
            scores = {}
            errors.append((sample.get("id"), str(exc)))
        sid = str(sample.get("id", "")).split("__")[-1]
        family_rows = _flatten_family(scores)
        for row in family_rows:
            detail_rows.append({"id": sid, "act": sample.get("act"), **extra, **row})
        scored_family = next(
            (r["family"] for r in family_rows if r.get("family") and r["family"] != "timing"),
            family,
        )
        numeric = [r for r in family_rows if r.get("family") == scored_family]
        merged = {"id": sid, "act": sample.get("act"), "family": scored_family, **extra}
        if numeric:
            keys = {k for r in numeric for k, v in r.items() if isinstance(v, (int, float))}
            for k in keys:
                vals = [r[k] for r in numeric if isinstance(r.get(k), (int, float))]
                if vals:
                    merged[k] = sum(vals) / len(vals)
            dirs = [r.get("sample_direction") for r in numeric if r.get("sample_direction")]
            if dirs:
                merged["sample_direction"] = max(set(dirs), key=dirs.count)
            gates = [r.get("gate") for r in numeric if r.get("gate")]
            if gates:
                merged["gate"] = max(set(gates), key=gates.count)
        summary_rows.append(merged)
        if scores:
            collapsed = {}
            for fam, metrics in scores.items():
                entries = metrics if isinstance(metrics, list) else [metrics]
                dicts = [e for e in entries if isinstance(e, dict)]
                if not dicts:
                    continue
                if len(dicts) == 1:
                    collapsed[fam] = dicts[0]
                    continue
                avg = {}
                keys = {k for e in dicts for k, v in e.items() if isinstance(v, (int, float))}
                for k in keys:
                    vals = [e[k] for e in dicts if isinstance(e.get(k), (int, float))]
                    avg[k] = sum(vals) / len(vals) if vals else None
                collapsed[fam] = avg
            per_record.append(("manim_play", sample.get("act") or "", collapsed))
    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(detail_rows),
        aggregate_family_scores(per_record) if per_record else {},
        errors,
    )


In [ ]:
judge_ready = setup_spectra_judge()
score_summary, score_detail, score_agg, score_errors = score_predictions(
    predictions, cfg=join_config, include_content=judge_ready,
)
if score_errors:
    print(f"{len(score_errors)} scoring issue(s):")
    for rid, msg in score_errors[:12]:
        print(f"  {rid}: {msg}")

summary_cols = [
    "id", "act", "family", "n_utt", "n_trig",
    "sample_direction", "gate",
    "precision", "purity", "iou",
    "timeliness", "content_raw", "content_decayed", "pauc",
    "far", "trap_rate", "silence_rate",
    "first_in_window_latency_s", "mean_latency_s", "time_error_s",
]
present = [c for c in summary_cols if c in score_summary.columns]
score_summary[present]

In [ ]:
from IPython.display import display

print("SPECTRA family aggregates (mean over samples; families are not mixed)")
agg_rows = [{"key": key, **metrics} for key, metrics in score_agg.items()]
agg_df = pd.DataFrame(agg_rows)
if not agg_df.empty:
    display(agg_df.set_index("key"))

act_metric_cols = [
    c for c in (
        "precision", "purity", "iou", "timeliness",
        "content_raw", "content_decayed", "pauc",
        "far", "trap_rate", "silence_rate",
        "first_in_window_latency_s",
    ) if c in score_summary.columns
]
if act_metric_cols:
    by_act = (
        score_summary.groupby("act", dropna=False)[act_metric_cols]
        .mean(numeric_only=True)
        .round(3)
    )
    print("\nPer-act means (SPECTRA family metrics + FAR / trap / silence)")
    display(by_act)

print("\nPer-trigger SPECTRA rows")
hide = {"n_utt", "n_trig", "far", "trap_rate", "silence_rate",
        "first_in_window_latency_s", "mean_in_window_latency_s"}
detail_show = [c for c in score_detail.columns if c not in hide]
display(score_detail[detail_show] if detail_show else score_detail)

plot_metrics = [
    c for c in ("precision", "purity", "timeliness", "content_raw", "pauc", "far", "silence_rate")
    if c in score_summary.columns and score_summary[c].notna().any()
]
if plot_metrics and not score_summary.empty:
    n = len(plot_metrics)
    fig, axes = plt.subplots(n, 1, figsize=(12, 2.2 * n), sharex=True)
    if n == 1:
        axes = [axes]
    x = list(score_summary["id"].astype(str))
    act_cat = score_summary["act"].astype("category")
    bar_colors = [plt.cm.tab10(i) for i in act_cat.cat.codes.to_numpy()]
    for ax, col in zip(axes, plot_metrics):
        ax.bar(x, score_summary[col].fillna(0), color=bar_colors, edgecolor="none")
        ax.set_ylabel(col)
        ymax = score_summary[col].max(skipna=True)
        ax.set_ylim(0, 1.05 if pd.isna(ymax) else max(1.05, float(ymax) * 1.1))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    axes[-1].set_xlabel("sample")
    axes[0].set_title("Per-sample SPECTRA scores and timing diagnostics", loc="left")
    acts = list(act_cat.cat.categories)
    axes[0].legend(
        handles=[Patch(color=plt.cm.tab10(i), label=a) for i, a in enumerate(acts)],
        loc="upper right", frameon=False, ncol=min(4, len(acts)),
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)